# Phase 5 — NB3 v5: Joint Evaluation (Category → Sentiment)

**Goal:** End-to-end evaluation: Stage 1 predict categories → Stage 2 predict sentiment.
Compare Phase 2a (W matrix retrieval) vs no-retrieval.

**Stage 1 checkpoint:** `stage1_r5_cataware_best.pt` (Cat-Aware Attention, val Cat F1=0.7243 @ ep26)

**Stage 2 checkpoints:**
- Retrieval: `stage2_phase2a_v2_best.pt` (W matrix, tau=0.1, augmented data, val Macro F1=0.7916)
- No-ret: `stage2_noret_best.pt` (augmented data, val Macro F1=0.6737)

**Input:**
- `lcminhc/p5-nb1-stage1` — Stage 1 ckpt + processed data
- `lcminhc/p5-nb2-stage2` — Stage 2 ckpts (phase2a + no-retrieval)
- `lcminhc/p3s2-embedding-flat` — embedding ckpt + FAISS index

**Output:** Metrics report (Category F1, Joint F1, Sentiment Acc|Correct Cat)

## 0. Setup

In [ ]:
!pip install -q transformers faiss-cpu lxml scikit-learn pyyaml

In [ ]:
import os
import sys
import json
import shutil

!git clone https://github.com/lucminhduc3108/Retrieval-ABSA.git /kaggle/working/repo
os.chdir('/kaggle/working/repo')
sys.path.insert(0, '/kaggle/working/repo')
print('Working dir:', os.getcwd())

In [ ]:
import torch
import gc
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
gc.collect()
torch.cuda.empty_cache()

In [ ]:
def find_input(name):
    for p in [f'/kaggle/input/{name}', f'/kaggle/input/datasets/lcminhc/{name}']:
        if os.path.exists(p):
            return p
    raise FileNotFoundError(f'Dataset {name} not found')

NB1 = find_input('p5-nb1-stage1')
NB2 = find_input('p5-nb2-stage2')
EMB = find_input('p3s2-embedding-flat')

print(f'NB1: {NB1} -> {os.listdir(NB1)}')
print(f'NB2: {NB2} -> {os.listdir(NB2)}')
print(f'EMB: {EMB} -> {os.listdir(EMB)[:5]}...')

# Stage 1 checkpoint
os.makedirs('checkpoints/stage1_r5_cataware', exist_ok=True)
shutil.copy(f'{NB1}/stage1_r5_cataware_best.pt', 'checkpoints/stage1_r5_cataware/best.pt')

# Stage 2 checkpoints — Phase 2a (W matrix) + no-retrieval
os.makedirs('checkpoints/stage2_phase2a', exist_ok=True)
os.makedirs('checkpoints/stage2_noret', exist_ok=True)
shutil.copy(f'{NB2}/stage2_phase2a_v2_best.pt', 'checkpoints/stage2_phase2a/best.pt')
shutil.copy(f'{NB2}/stage2_noret_best.pt', 'checkpoints/stage2_noret/best.pt')

# Embedding checkpoint only (NOT old index — rebuild below)
os.makedirs('checkpoints/embedding', exist_ok=True)
shutil.copy(f'{EMB}/embedding_best.pt', 'checkpoints/embedding/best.pt')

# Processed data
os.makedirs('data/processed', exist_ok=True)
shutil.copy(f'{NB1}/category_detection.jsonl', 'data/processed/category_detection.jsonl')
shutil.copy(f'{NB1}/sentiment_records.jsonl', 'data/processed/sentiment_records.jsonl')
shutil.copy(f'{NB1}/sentiment_records.jsonl', 'data/processed/sentiment_records_aug.jsonl')

# Rebuild FAISS index from Phase 5 data (must match NB2 training index)
os.makedirs('indexes', exist_ok=True)
!python scripts/03_build_index.py \
    --embedding_ckpt checkpoints/embedding/best.pt \
    --input data/processed/sentiment_records.jsonl \
    --out_dir indexes/

print('\nAll files wired:')
for p in ['checkpoints/stage1_r5_cataware/best.pt', 'checkpoints/stage2_phase2a/best.pt',
          'checkpoints/stage2_noret/best.pt', 'checkpoints/embedding/best.pt',
          'indexes/train.faiss', 'data/processed/category_detection.jsonl',
          'data/processed/sentiment_records.jsonl',
          'data/processed/sentiment_records_aug.jsonl']:
    size = os.path.getsize(p) / 1e6
    print(f'  {p}: {size:.1f} MB')

## 1. Evaluate WITH Retrieval (Phase 2a — W matrix)

In [ ]:
!python scripts/05_evaluate_joint.py \
    --stage1_ckpt checkpoints/stage1_r5_cataware/best.pt \
    --stage2_ckpt checkpoints/stage2_phase2a/best.pt \
    --embedding_ckpt checkpoints/embedding/best.pt \
    --index_dir indexes/ \
    --stage1_config configs/stage1_r5_cataware.yaml \
    --stage2_config configs/stage2_phase2a_v2.yaml \
    --pred_strategy all

In [ ]:
if os.path.exists('logs/joint_eval_results.md'):
    shutil.copy('logs/joint_eval_results.md', 'logs/joint_eval_retrieval.md')
    print('Retrieval results saved')

## 2. Evaluate WITHOUT Retrieval

In [ ]:
gc.collect()
torch.cuda.empty_cache()

!python scripts/05_evaluate_joint.py \
    --stage1_ckpt checkpoints/stage1_r5_cataware/best.pt \
    --stage2_ckpt checkpoints/stage2_noret/best.pt \
    --stage1_config configs/stage1_r5_cataware.yaml \
    --stage2_config configs/stage2_noret.yaml \
    --no_retrieval \
    --pred_strategy all

In [ ]:
if os.path.exists('logs/joint_eval_results.md'):
    shutil.copy('logs/joint_eval_results.md', 'logs/joint_eval_noret.md')
    print('No-retrieval results saved')

## 3. Results Comparison

In [ ]:
print('=' * 60)
print('PHASE 2a RETRIEVAL (W matrix)')
print('=' * 60)
if os.path.exists('logs/joint_eval_retrieval.md'):
    with open('logs/joint_eval_retrieval.md') as f:
        print(f.read())

print('\n' + '=' * 60)
print('NO-RETRIEVAL BASELINE (augmented data)')
print('=' * 60)
if os.path.exists('logs/joint_eval_noret.md'):
    with open('logs/joint_eval_noret.md') as f:
        print(f.read())

print('\n' + '=' * 60)
print('REFERENCE: NB3 v4 (Cat-Aware R5 + Stage 2 Run 2)')
print('=' * 60)
print('global (0.80): Cat F1=0.6962 | No-Ret Joint F1=0.6235 | No-Ret Sent Acc|CC=0.9006')
print('global (0.80): Cat F1=0.6962 | Ret    Joint F1=0.6139 | Ret    Sent Acc|CC=0.8867')

## 4. Save Outputs

In [ ]:
output_dir = '/kaggle/working/outputs_p5_nb3'
os.makedirs(output_dir, exist_ok=True)

if os.path.exists('logs'):
    shutil.copytree('logs', f'{output_dir}/logs', dirs_exist_ok=True)
    print('logs/ copied')

print(f'\nOutputs saved to {output_dir}')